In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("alchimistvq/xauusd-historical-data-1m")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\alchimistvq\xauusd-historical-data-1m\versions\1


In [10]:
import os

for p in os.listdir(path):
    print(p)
    path += "/" + p

XAUUSD_1m.csv


In [11]:
import pandas as pd

# Read the CSV and normalize the time column to a consistent UTC datetime.
df = pd.read_csv(path)
df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
df = df.dropna(subset=["time"]).reset_index(drop=True)


In [12]:
print (f"row: {len(df)}")
print (f"column: {len(df.columns)}")

row: 677920
column: 6


In [13]:
df.isna()

,time,open,high,low,close,volume
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,False
3,False,False,False,False,False,False
4,False,False,False,False,False,False
...,...,...,...,...,...,...
677915,False,False,False,False,False,False
677916,False,False,False,False,False,False
677917,False,False,False,False,False,False
677918,False,False,False,False,False,False


In [14]:
print(df.head(5))
print(df.tail(5))

                       time     open     high      low    close  volume
0 2024-01-02 05:01:00+00:00  2062.14  2069.79  2062.14  2069.79       4
1 2024-01-02 05:02:00+00:00  2069.79  2069.79  2069.49  2069.53      66
2 2024-01-02 05:03:00+00:00  2069.53  2069.55  2069.39  2069.44      95
3 2024-01-02 05:04:00+00:00  2069.44  2069.67  2069.40  2069.64      93
4 2024-01-02 05:05:00+00:00  2069.64  2069.90  2069.56  2069.89      75
                            time     open     high      low    close  volume
677915 2025-11-28 20:56:00+00:00  4238.82  4239.17  4238.78  4239.14      24
677916 2025-11-28 20:57:00+00:00  4239.14  4239.20  4238.90  4239.20      28
677917 2025-11-28 20:58:00+00:00  4239.20  4239.26  4239.20  4239.25       7
677918 2025-11-28 20:59:00+00:00  4239.25  4239.41  4238.89  4239.39      67
677919 2025-11-28 21:00:00+00:00  4239.39  4239.41  4239.39  4239.41       3


In [15]:
print(df.describe())
print(df.info())

                open           high            low          close  \
count  677920.000000  677920.000000  677920.000000  677920.000000   
mean     2853.589797    2854.098612    2853.074964    2853.593077   
std       589.704319     589.916701     589.481195     589.705881   
min      1984.940000    1985.710000    1984.070000    1984.940000   
25%      2371.320000    2371.730000    2370.890000    2371.320000   
50%      2681.340000    2681.740000    2680.905000    2681.345000   
75%      3329.670000    3330.200000    3329.120000    3329.670000   
max      4380.640000    4381.510000    4377.780000    4380.640000   

              volume  
count  677920.000000  
mean      381.821774  
std       388.393141  
min         1.000000  
25%       141.000000  
50%       263.000000  
75%       480.000000  
max      7442.000000  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 677920 entries, 0 to 677919
Data columns (total 6 columns):
 #   Column  Non-Null Count   Dtype              
---  ------

In [16]:
import numpy as np

# RSI 14-periode dari harga close.
df = df.sort_values("time").reset_index(drop=True)

period = 14
delta = df["close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()

rs = avg_gain / avg_loss
df["rsi"] = 100 - (100 / (1 + rs))
df.loc[avg_loss == 0, "rsi"] = 100
df.loc[avg_gain == 0, "rsi"] = 0

In [17]:
df.to_csv("gold_1m.csv", index=False)